# DevUploads Direct-Link Downloader & Uploader

This notebook:
1. Downloads a file from a direct link
2. Uploads it to [DevUploads](https://devuploads.com/api) using their Upload API

Fill in your API key and the direct link in the cell below, then run all cells.

In [ ]:
!pip install -q requests

In [ ]:
# @title DevUploads Direct-Link Downloader & Uploader
# @markdown Fill in your API key and the direct download link, then run.

import requests
import os
from urllib.parse import urlparse, unquote

API_KEY = "YOUR_DEVUPLOADS_API_KEY"  # @param {type:"string"}
DIRECT_LINK = "https://example.com/path/to/file.zip"  # @param {type:"string"}
DOWNLOAD_DIR = "/content/downloads"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

In [ ]:
def get_filename_from_url(url, response):
    # Try Content-Disposition header first
    cd = response.headers.get("content-disposition")
    if cd and "filename=" in cd:
        fname = cd.split("filename=")[-1].strip('"; ')
        if fname:
            return unquote(fname)
    # Fallback to URL path
    path = urlparse(url).path
    fname = os.path.basename(path)
    return unquote(fname) if fname else "downloaded_file"


def download_file(url, dest_dir):
    print(f"\u2b07\ufe0f  Downloading: {url}")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        filename = get_filename_from_url(url, r)
        filepath = os.path.join(dest_dir, filename)
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(filepath, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        pct = downloaded / total * 100
                        print(f"\r   {pct:5.1f}% ({downloaded/1e6:.1f} MB / {total/1e6:.1f} MB)", end="")
        print(f"\n\u2705  Saved to {filepath}")
        return filepath


def get_upload_server(api_key):
    print("\U0001f310  Getting upload server...")
    resp = requests.get(
        "https://devuploads.com/api/upload/server",
        params={"key": api_key},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    if data.get("status") != 200:
        raise RuntimeError(f"Failed to get upload server: {data}")
    upload_url = data["result"]
    print(f"\u2705  Upload server: {upload_url}")
    return upload_url


def upload_to_devuploads(filepath, api_key):
    upload_url = get_upload_server(api_key)

    # sess_id is required by the upload form. DevUploads issues one per
    # upload session; if not returned separately, an empty/placeholder
    # value is accepted by most servers, but include a random one to be safe.
    import uuid
    sess_id = uuid.uuid4().hex[:16]

    filename = os.path.basename(filepath)
    print(f"\u2b06\ufe0f  Uploading {filename} to DevUploads...")

    with open(filepath, "rb") as f:
        files = {"file": (filename, f)}
        data = {"sess_id": sess_id}
        resp = requests.post(upload_url, data=data, files=files, timeout=None)

    resp.raise_for_status()

    try:
        result = resp.json()
    except ValueError:
        print("\u26a0\ufe0f  Non-JSON response, raw text:")
        print(resp.text)
        return None

    print("\u2705  Upload complete!")
    print(result)
    return result

In [ ]:
# --- Run pipeline ---
local_path = download_file(DIRECT_LINK, DOWNLOAD_DIR)
result = upload_to_devuploads(local_path, API_KEY)

## Notes

- **`sess_id`**: DevUploads' public docs show a hardcoded `sess_id` in their sample form (`3qr5wkukoy31pd1g`) but don't fully document how to obtain one tied to your account. This notebook generates a random one — if uploads are rejected, inspect a manual upload on their site (browser dev tools → Network tab) to find the real value, or contact DevUploads support.
- **Large files**: Colab's disk and runtime are limited (free tier is typically ~50–100GB disk, and sessions time out after inactivity), so very large files may need a Pro runtime or chunked handling.
- **API key**: get yours from your DevUploads account dashboard.